# 옛 예측 파일을 실행 폴더로 옮기기 (CPU)

`vqa_runs` 바로 아래에 있는 옛 노트북의 예측 파일(`preds_<모델>__<해상도>__4bit__valid1000.csv` / `…__test6714.csv`)은 앙상블 노트북들이 찾지 못합니다. 이 노트북이 이 파일들을 표준 실행 폴더로 옮깁니다.
- 옮긴 뒤 **ens_search · manual_ensemble · E14 · final_pick을 다시 실행하면 자동으로 멤버로 들어갑니다.**
- 원래 파일은 건드리지 않고 복사만 합니다.

**옮기기 전에 확인하는 것** (하나라도 걸리면 그 파일은 옮기지 않음)
1. 검증 파일의 문항이 팀 공통 검증 1,000문항과 정확히 같은가.
   - 다르면 다른 분할로 학습한 모델이라, 우리 검증 문항을 학습에 썼을 수 있습니다.
2. test 파일에 6,714문항이 다 있는가.
3. 확률 열(`p_a`~`p_d`)이 있고 값이 정상인가.
4. 검증이 99%를 넘지 않는가. 넘으면 검증 누출을 의심합니다.
5. 기존 실행의 복사본이 아닌가.
   - 검증 답이 99.9% 이상 같고 확률도 거의 같으면 같은 모델로 보고 건너뜁니다. 같은 모델을 두 번 넣으면 표만 두 번 세게 됩니다.
   - 가장 비슷한 기존 실행 3개와의 일치율도 보여 줍니다.


## 1. 설정


In [ ]:
DRIVE_ZIP    = "/content/drive/MyDrive/ssafy-16-2-ai.zip"
ZIP_PASSWORD = "YOUR_ZIP_PASSWORD"
DATA_DIR     = "/content"
OUT_DIR      = "/content/drive/MyDrive/vqa_runs"
ENSEMBLE_SUBDIR = "ensembles"
SEED, VAL_SIZE = 42, 1000
N_PUBLIC = 3357

SRC_SUBDIR = ""        # 옛 파일이 있는 폴더 (OUT_DIR 기준). 지금은 vqa_runs 바로 아래라 ""
# (새 실행 이름, 검증 파일, test 파일, 해상도, 기반 모델 ID, 같은 계열의 기존 실행 — 있으면 그 train_log의 MODEL_ID를 그대로 씀)
IMPORTS = [
    ("q35_27b_old_up15", "preds_Qwen3.5-27B__up1.5__4bit__valid1000.csv", "preds_Qwen3.5-27B__up1.5__4bit__test6714.csv", "up1.5", "Qwen/Qwen3.5-27B", "q35_27b_r16_native"),
    ("q35_27b_old_up20", "preds_Qwen3.5-27B__up2.0__4bit__valid1000.csv", "preds_Qwen3.5-27B__up2.0__4bit__test6714.csv", "up2.0", "Qwen/Qwen3.5-27B", "q35_27b_r16_native"),
    ("q3vl_32b_old_up15", "preds_Qwen3-VL-32B__up1.5__4bit__valid1000.csv", "preds_Qwen3-VL-32B__up1.5__4bit__test6714.csv", "up1.5", "Qwen/Qwen3-VL-32B-Instruct", "q3vl_32b_r16_native_full"),
]
MIN_VAL_ACC = 90.0     # 이보다 낮으면 고장으로 보고 옮기지 않음
SUSPECT_ACC = 99.0     # 이보다 높으면 검증 누출 의심 → 옮기지 않음
DUP_AGREE = 99.9       # 기존 실행과 검증 답 일치율(%)이 이 이상이고
DUP_MAXDIFF = 1e-3     #   확률 차이가 이 이하면 복사본으로 보고 건너뜀
ADD_DUPLICATES = False # True면 복사본이어도 옮김 (보통 False)
OVERWRITE = False      # True면 이미 있는 같은 이름 폴더를 덮어씀


## 2. 데이터 (CSV만 꺼냄) · 팀 공통 검증 분할


In [ ]:
import os, re, json, math, hashlib, subprocess
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount("/content/drive"); IN_COLAB = True
except ImportError:
    IN_COLAB = False
_need = ["train.csv", "test.csv", "sample_submission.csv"]
def _have(): return all(os.path.exists(os.path.join(DATA_DIR, f)) for f in _need)
if IN_COLAB and not _have():
    print("압축에서 CSV 3개만 꺼냅니다...")
    subprocess.run(["7z", "e", "-y", f"-p{ZIP_PASSWORD}", f"-o{DATA_DIR}", DRIVE_ZIP] + _need + ["-r"], capture_output=True, text=True)
    if not _have():
        print("전체 압축 해제로 다시 시도합니다 (몇 분 걸립니다)")
        r = subprocess.run(["7z", "x", "-y", f"-p{ZIP_PASSWORD}", f"-o{DATA_DIR}", DRIVE_ZIP], capture_output=True, text=True)
        assert r.returncode == 0, r.stderr[-800:]
assert _have(), f"{_need}가 {DATA_DIR}에 없습니다 → DATA_DIR / 압축 경로를 확인하세요"
assert os.path.isdir(OUT_DIR), f"{OUT_DIR}가 없습니다 → OUT_DIR을 확인하세요"
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sub_df = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
train_df["answer"] = train_df["answer"].astype(str).str.strip().str.lower()
assert list(test_df.id) == list(sub_df.id), "test.csv와 sample_submission.csv의 id 순서가 다릅니다"
LETTERS = "abcd"
def make_split(df, val_size=VAL_SIZE, seed=SEED):      # stage2 노트북과 같은 함수
    rng = np.random.RandomState(seed); val_idx = []
    for letter in "abcd":
        idx = np.where(df.answer.values == letter)[0]; rng.shuffle(idx); val_idx.extend(idx[: val_size // 4].tolist())
    val_idx = sorted(val_idx); mask = np.zeros(len(df), dtype=bool); mask[val_idx] = True
    return df[~mask].reset_index(drop=True), df[mask].reset_index(drop=True)
tr_df, val_df = make_split(train_df)
assert hashlib.md5(",".join(val_df.id.tolist()).encode()).hexdigest() == "df05e4bc19d27ebceeef2570672149c2", "검증 분할이 팀 공통 분할과 다릅니다"
VAL_IDS, TEST_IDS = val_df.id.tolist(), test_df.id.tolist()
Y_VAL = val_df.answer.map(LETTERS.index).to_numpy()
ENS_DIR = os.path.join(OUT_DIR, ENSEMBLE_SUBDIR)
print(f"✅ 데이터 OK | 검증 {len(VAL_IDS):,} (팀 공통 분할) | test {len(TEST_IDS):,}")


## 3. 예측 파일 읽기 함수 (앙상블 노트북들과 같음)


In [ ]:
_CACHE, _TCACHE = {}, {}
def _probs(d): return d[["p_a", "p_b", "p_c", "p_d"]].to_numpy(dtype=float)
def _load_preds(path, ids):
    d = pd.read_csv(path).drop_duplicates("id", keep="last").set_index("id")
    miss = [i for i in ids if i not in d.index]
    assert not miss, f"{os.path.basename(path)}에 없는 문항 {len(miss)}개 (예: {miss[:3]})"
    return d.loc[ids]
def read_log(run):
    p = os.path.join(OUT_DIR, run, "train_log.json")
    try: return json.load(open(p, encoding="utf-8")) if os.path.exists(p) else {}
    except Exception: return {}
def member_file(run, preset, split):
    # 검증: preds_final_ep{N}__{preset}__1000.csv / test: preds_final_ep{N}_test__{preset}__6714.csv (epoch는 E7과 같은 규칙으로 고름)
    d = os.path.join(OUT_DIR, run)
    if not os.path.isdir(d): return None, "폴더 없음"
    rv = re.compile(rf"^preds_final_ep(\d+)__{re.escape(preset)}__1000\.csv$")
    rt = re.compile(rf"^preds_final_ep(\d+)_test__{re.escape(preset)}__(\d+)\.csv$")
    cv, ct = {}, {}
    for f in os.listdir(d):
        m = rv.match(f)
        if m: cv[int(m.group(1))] = f; continue
        m = rt.match(f)
        if m and int(m.group(2)) == len(TEST_IDS): ct[int(m.group(1))] = f
    base = cv or ct
    if not base: return None, "예측 파일 없음"
    ea = {int(k): v for k, v in (read_log(run).get("epoch_acc") or {}).items()}
    ep = max(ea, key=lambda e: (ea[e], e)) if ea else None
    ep = ep if ep in base else max(base)
    pool = cv if split == "val" else ct
    if ep not in pool: return None, f"ep{ep} {split} 파일 없음"
    return os.path.join(d, pool[ep]), f"ep{ep}"
def load(run, preset, split):
    key = (run, preset, split)
    if key not in _CACHE:
        f, why = member_file(run, preset, split)
        if f is None: raise FileNotFoundError(f"{run}@{preset} {split}: {why}")
        d = _load_preds(f, VAL_IDS if split == "val" else TEST_IDS)
        _CACHE[key] = (_probs(d), f, float(d["letter_mass"].mean()) if "letter_mass" in d.columns else float("nan"))
    return _CACHE[key][0]
def file_answers(fname):
    p = os.path.join(ENS_DIR, fname)
    if not os.path.exists(p): return None
    s = pd.read_csv(p).drop_duplicates("id", keep="last").set_index("id").reindex(TEST_IDS)["answer"]
    s = s.astype(str).str.strip().str.lower().map({L: i for i, L in enumerate(LETTERS)})
    return None if s.isna().any() else s.to_numpy().astype(int)
T_GRIDS = {"T70": np.exp(np.linspace(np.log(0.25), np.log(8.0), 70)),   # 온도 격자 두 가지 (Stage 3가 어느 쪽이었든 찾도록)
           "T28": np.linspace(0.3, 3.0, 28)}
def temp_scale(p, T):
    q = np.clip(p, 1e-12, 1.0) ** (1.0 / T); return q / q.sum(1, keepdims=True)
def fit_T(run, preset, grid):
    # 멤버 하나의 온도: 원래(검증 제외) 멤버의 검증 예측으로 NLL 최소
    key = (run, preset, grid)
    if key not in _TCACHE:
        p, i = load(run, preset, "val"), np.arange(len(Y_VAL))
        nll = [-np.mean(np.log(temp_scale(p, T)[i, Y_VAL])) for T in T_GRIDS[grid]]
        _TCACHE[key] = float(T_GRIDS[grid][int(np.argmin(nll))])
    return _TCACHE[key]
def ens_scores(members, weights, how="prob", cal=None, split="test", swap=None, mix=False):
    # how: "prob" 확률 가중 평균 / "log" 로그확률 가중 평균. swap: {원래 멤버: 전체 데이터 멤버} → test 확률만 교체(mix면 둘의 평균)
    w = np.asarray(weights, dtype=float); assert len(w) == len(members), "members와 weights 개수가 다릅니다"
    w = w / w.sum(); out = 0.0
    for wi, (run, pre) in zip(w, members):
        P = load(run, pre, split)
        if swap and run in swap and split == "test":
            Pf = load(swap[run], pre, "test"); P = (P + Pf) / 2 if mix else Pf
        if cal: P = temp_scale(P, fit_T(run, pre, cal))
        out = out + wi * (P if how == "prob" else np.log(np.clip(P, 1e-12, 1.0)))
    return out
def vote_of(arrs):
    A = np.stack(arrs, 1); cnt = np.stack([(A == k).sum(1) for k in range(4)], 1)
    return cnt.argmax(1)             # 동률(셋 다 다름)이면 알파벳 앞 글자 — pandas mode(axis=1)[0]와 같음
def nll(P, y): return float(-np.mean(np.log(np.clip(P[np.arange(len(y)), y], 1e-12, 1.0))))
def paired(a_old, a_new, y):
    c0, c1 = a_old == y, a_new == y
    f, b = int((~c0 & c1).sum()), int((c0 & ~c1).sum()); return f, b, f - b, int((a_old != a_new).sum())
CFG_DEFAULTS = {"TRAIN_SEED": 42, "MAX_STEPS": None, "EXTRA_TRAIN_CSV": None, "IMG_AUG": False, "TRAIN_PRESETS": None,
                "AUX_GEN_CSV": None, "AUX_GEN_EVERY": 5, "AUX_GEN_WEIGHT": 1.0}     # 예전 train_log에 없는 항목 = 이 값 (stage2 이어하기 규칙과 같음)
def cfg_val(c, k):
    v = c[k] if k in c else CFG_DEFAULTS.get(k)
    if v is None: v = CFG_DEFAULTS.get(k)
    return None if (isinstance(v, (list, tuple)) and not v) else v
def rname(members, how, cal):
    return "+".join(r.split("_")[0] + ("dev" if r.endswith("_dev") else "") for r, _ in members) + f" | {'확률' if how == 'prob' else '로그확률'} 평균" + (f" | 온도 {cal}" if cal else "")
def score_of(count): return math.floor(count / N_PUBLIC * 1e5) / 1e5          # 리더보드 표시(소수 5자리 버림)
def count_of(score): return int(round(score * N_PUBLIC))
print("✅ 함수 준비 완료")


## 4. 기존 실행의 검증 예측 모으기 (복사본 확인용)


In [ ]:
EXIST = {}
for run in sorted(os.listdir(OUT_DIR)):
    d = os.path.join(OUT_DIR, run)
    if not os.path.isdir(d): continue
    for pre in sorted({m.group(2) for f in os.listdir(d) if (m := re.match(r"^preds_final_ep(\d+)__(.+)__1000\.csv$", f))}):
        try: EXIST[(run, pre)] = load(run, pre, "val")
        except Exception: pass
print(f"기존 실행의 검증 예측 {len(EXIST)}개 (실행 {len({r for r, _ in EXIST})}개)")
_vf = os.path.join(OUT_DIR, "valid_ids.csv")
if os.path.exists(_vf):
    _v = pd.read_csv(_vf); _col = "id" if "id" in _v.columns else _v.columns[0]; _ids = set(_v[_col].astype(str))
    _same = _ids == set(VAL_IDS) or {i + ".jpg" for i in _ids} == set(VAL_IDS)
    print(f"vqa_runs/valid_ids.csv ({len(_ids)}개): 팀 공통 검증 분할과 " + ("같음 ✅" if _same else "다름 ⚠️ — 옛 노트북이 다른 분할을 썼을 수 있음"))


## 5. 파일마다 확인


In [ ]:
LETTERS4 = ["p_a", "p_b", "p_c", "p_d"]
def read_old(path, ids):
    d = pd.read_csv(path)
    miss = [c for c in ["id"] + LETTERS4 if c not in d.columns]
    if miss: return None, f"열 없음 {miss} (있는 열: {list(d.columns)[:12]})"
    d["id"] = d["id"].astype(str); want = set(ids)
    if d["id"].isin(want).mean() < 0.5:                                        # id 모양이 다르면 .jpg 붙이기/떼기
        for fix in (lambda s: s + ".jpg", lambda s: s[:-4] if s.endswith(".jpg") else s):
            alt = d["id"].map(fix)
            if alt.isin(want).mean() > 0.99: d["id"] = alt; break
    d = d.drop_duplicates("id", keep="last").set_index("id")
    missing, extra = [i for i in ids if i not in d.index], sorted(set(d.index) - want)
    if missing or extra: return None, f"문항이 다름: 없는 문항 {len(missing)}개 (예: {missing[:3]}) · 더 있는 문항 {len(extra)}개 (예: {extra[:3]})"
    d = d.loc[ids]; P = d[LETTERS4].to_numpy(float)
    if not np.isfinite(P).all() or (P < 0).any(): return None, "확률에 NaN/음수가 있음"
    s = P.sum(1)
    if np.abs(s - 1).max() > 1e-3:
        if (s <= 0).any(): return None, "확률 합이 0인 문항이 있음"
        d[LETTERS4] = P / s[:, None]; print(f"    (확률 합이 1이 아닌 문항이 있어 네 보기로 다시 나눔: 합 {s.min():.3f}~{s.max():.3f})")
    if "pred" not in d.columns: d["pred"] = [LETTERS[i] for i in d[LETTERS4].to_numpy().argmax(1)]
    return d, None

READY, _rows = [], []
for name, fv, ft, pre, mid, like in IMPORTS:
    pv, pt = os.path.join(OUT_DIR, SRC_SUBDIR, fv), os.path.join(OUT_DIR, SRC_SUBDIR, ft)
    row = {"새 실행": name, "해상도": pre}
    print(f"▶ {name} ← {fv}")
    def _no(short, detail): row["판정"] = "❌ " + short; _rows.append(row); print(f"   ❌ {short} — {detail}")
    if not (os.path.exists(pv) and os.path.exists(pt)):
        _no("파일 없음", " · ".join(f for f, ok in ((pv, os.path.exists(pv)), (pt, os.path.exists(pt))) if not ok)); continue
    dv, why = read_old(pv, VAL_IDS)
    if dv is None: _no("검증 파일 문제", why); continue
    dt, why = read_old(pt, TEST_IDS)
    if dt is None: _no("test 파일 문제", why); continue
    P = dv[LETTERS4].to_numpy(float); acc = float((P.argmax(1) == Y_VAL).mean() * 100)
    row.update({"검증(%)": round(acc, 2), "NLL": round(nll(P, Y_VAL), 4), "letter_mass": round(float(dv["letter_mass"].mean()), 4) if "letter_mass" in dv.columns else None})
    sims = sorted(((float((P.argmax(1) == Q.argmax(1)).mean() * 100), float(np.abs(P - Q).max()), r, p) for (r, p), Q in EXIST.items() if r != name),   # 이미 옮긴 자기 폴더는 빼고
                  key=lambda t: (-t[0], t[1], t[2], t[3]))           # 답 일치율 높은 순 → 확률 차이 작은 순
    row["가장 비슷한 기존 실행 (검증 답 일치%)"] = " · ".join(f"{r}@{p} {a:.1f}" for a, _, r, p in sims[:3])
    dup = [(r, p) for a, dmax, r, p in sims if a >= DUP_AGREE and dmax <= DUP_MAXDIFF]
    if acc < MIN_VAL_ACC: row["판정"] = f"❌ 검증 < {MIN_VAL_ACC}% (고장)"
    elif acc > SUSPECT_ACC: row["판정"] = f"❌ 검증 > {SUSPECT_ACC}% (검증 누출 의심)"
    elif dup and not ADD_DUPLICATES: row["판정"] = f"⏭ 복사본 ({dup[0][0]}@{dup[0][1]}와 같은 예측) → 건너뜀"
    else:
        fam = mid
        lg = (read_log(like).get("config") or {}) if like else {}
        if lg.get("MODEL_ID"): fam = str(lg["MODEL_ID"])
        READY.append((name, fv, ft, pre, fam, dv, dt, pv, pt, acc))
        row["판정"] = ("✅ 이미 옮겨져 있음" if os.path.isdir(os.path.join(OUT_DIR, name)) else "✅ 옮길 수 있음") + (" (복사본이지만 ADD_DUPLICATES=True)" if dup else "")
        row["계열 (MODEL_ID)"] = fam
    _rows.append(row); print("   " + row["판정"] + f" | 검증 {acc:.2f}%")
print()
print(pd.DataFrame(_rows).to_string(index=False))


## 6. 실행 폴더로 옮기기


In [ ]:
import datetime
def md5file(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()
DONE = []
for name, fv, ft, pre, fam, dv, dt, pv, pt, acc in READY:
    d = os.path.join(OUT_DIR, name)
    if os.path.exists(d) and os.listdir(d) and not OVERWRITE:
        print(f"⏭ {name}: 폴더가 이미 있음 → 그대로 둠 (다시 만들려면 OVERWRITE=True)"); DONE.append(name); continue
    os.makedirs(d, exist_ok=True)
    cols = ["id", "pred"] + LETTERS4 + [c for c in ("letter_mass", "img_tokens") if c in dv.columns]
    dv.reset_index()[[c for c in cols if c in dv.reset_index().columns]].to_csv(os.path.join(d, f"preds_final_ep1__{pre}__1000.csv"), index=False)
    dt.reset_index()[[c for c in cols if c in dt.reset_index().columns]].to_csv(os.path.join(d, f"preds_final_ep1_test__{pre}__{len(TEST_IDS)}.csv"), index=False)
    log = {"config": {"MODEL_ID": fam, "TRAIN_ON_FULL": False, "PRECISION": "4bit", "EVAL_PRESET": pre,
                      "IMPORTED_FROM": {"val": os.path.join(SRC_SUBDIR, fv), "test": os.path.join(SRC_SUBDIR, ft)}},
           "epoch_acc": {"1": round(acc / 100, 4)},
           "imported": {"when": datetime.datetime.now().isoformat(timespec="seconds"), "val_md5": md5file(pv), "test_md5": md5file(pt),
                        "note": "옛 노트북 예측 파일을 import_old_preds.ipynb로 옮김 (예측값은 그대로, 문항 순서만 맞춤)"}}
    json.dump(log, open(os.path.join(d, "train_log.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    DONE.append(name); print(f"✅ {name}/ 만듦: preds_final_ep1__{pre}__1000.csv · preds_final_ep1_test__{pre}__{len(TEST_IDS)}.csv · train_log.json")


## 7. 앙상블 노트북이 읽는 방식으로 다시 읽어 확인


In [ ]:
_CACHE.clear()
for name in DONE:
    pre = next(p for n, _, _, p, *_ in IMPORTS if n == name)
    fv_, why_v = member_file(name, pre, "val"); ft_, why_t = member_file(name, pre, "test")
    assert fv_ and ft_, f"{name}: 표준 이름으로 못 찾음 ({why_v} / {why_t})"
    Pv, Pt = load(name, pre, "val"), load(name, pre, "test")
    fam = (read_log(name).get("config") or {}).get("MODEL_ID")
    print(f"✅ {name}@{pre}: 검증 {(Pv.argmax(1) == Y_VAL).mean() * 100:.2f}% · test {len(Pt):,}문항 · 계열 {fam}")
print("\n다음: ens_search.ipynb / manual_ensemble.ipynb를 처음부터 다시 실행하면 이 실행들이 멤버로 들어갑니다 (4번 셀 '멤버 N개'가 늘어남).")
